<a href="https://colab.research.google.com/github/AcSsalazar/the-color-of-emotions/blob/main/Notebooks-EN/4.5-transfer-learning-pytorch-tensors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Transfer Learning and Fine-tuning with ResNet-18 and EfficientNet-B0 using PyTorch tensors

### Introduction:

**3D tensors:** Data processed directly with PyTorch from **Mel spectrograms**, **first-order delta spectrograms**, and **Cochlear spectrograms**. This yields an input tensor with shape `[3, Frequency, Time in frames]`, with dimensions `[3, 60, 51]`.

In [ ]:
# Imports
#----------------------------------------------------------------
import os
import gc
import numpy as np
import glob
import torch
import random
import torch.nn as nn
import seaborn as sns
import torch.optim as optim
import matplotlib.pyplot as plt
#----------------------------------------------------------------
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
from torchvision import datasets, transforms, models
from torchaudio import transforms as T
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from torch.cuda.amp import GradScaler, autocast
from google.colab import drive

In [ ]:
# Instalacion de Weights & Biases

!pip install wandb -q

# Import y Loging

import wandb; wandb.login()

In [ ]:
drive.mount('/content/drive')

In [ ]:
# Seeding for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Copy the entire features folder from Drive to Colab's ultra-fast disk
#!cp -r /content/drive/MyDrive/ravdess_images_02/ /content/features_local
os.makedirs('/content/', exist_ok=True)
# Optional: If you have a .zip file in Drive, it is EVEN FASTER to copy the .zip and unzip it locally:
!cp -r /content/drive/MyDrive/split_pytorch_tensors/ /content

In [ ]:
# Configurations and paths
BASE_DIR_TENSOR = '/content/split_pytorch_tensors'
MODELS_SAVE_DIR = '/content/saved_models_TL_Pytorch'
os.makedirs(MODELS_SAVE_DIR, exist_ok=True)
# Default batch = 64
BATCH_SIZE = 64
# Flag for unprepared data
already_normalized = True

In [ ]:
# Print root_dir content:
tensor_content = (os.listdir(BASE_DIR_TENSOR))
print(f"Contenido de {BASE_DIR_TENSOR}: {tensor_content}")

### Dataloaders with class names and metadata

Because we saved a pack with metadata, where we created a dictionary mapping indices to class names, we can create a custom class that extracts our tensors with their respective class names and dimensions for each dataset split.

### Normalization and augmentation strategy

**Normalization:**  
The tensors exported in notebook `3.2` were already normalized with **per-sample and per-channel z-score** (`zscore_per_channel`). That is why `already_normalized = True` and no second dataset-level normalization is applied here. Applying `CHANNEL_MEAN/CHANNEL_STD` on top of per-sample z-score would introduce redundant double normalization that can distort the input distribution.

**Online augmentation:**  
Notebook `3.2` already added extra `surprised` samples with **noise** and **time shift** during export. Consequently, here we use **SpecAugment only** (FrequencyMasking + TimeMasking) as online augmentation, exclusively on the training split. Noise and shift are not repeated to avoid cumulative augmentation.

**WeightedRandomSampler:**  
Since `surprised` is already over-represented due to offline export augmentation, `WeightedRandomSampler` is disabled by default (`use_weighted_sampler=False`). It can be enabled if tensors are regenerated without offline augmentation.

In [ ]:
SPLIT_FILES = {
    'train': 'train_tensors.pt',
    'val': 'val_tensors.pt',
    'test': 'test_tensors.pt',
}

def load_pack(split_name: str):
    path = os.path.join(BASE_DIR_TENSOR, SPLIT_FILES[split_name])
    if not os.path.exists(path):
        raise FileNotFoundError(f'File does not exist: {path}')
    return torch.load(path, map_location='cpu', weights_only=False)

# Load the packs (they are not pure tensors yet)
train_pack = load_pack('train')
val_pack = load_pack('val')
test_pack = load_pack('test')

# Verify the classes in the dictionary inside the pack
class_to_idx = train_pack['class_to_idx']
idx_to_class = {v: k for k, v in class_to_idx.items()}
class_names = [idx_to_class[i] for i in sorted(idx_to_class.keys())]
print('Clases:', class_names)
print('Shape train:', tuple(train_pack['x'].shape))

# CLASS_WEIGHTS is always calculated (it depends on labels, not normalization)
def compute_class_weights(labels: torch.Tensor, num_classes: int):
    counts = torch.bincount(labels, minlength=num_classes).float()
    weights = counts.sum() / (counts * num_classes)
    return weights

CLASS_WEIGHTS = compute_class_weights(train_pack['y'], len(class_names))
print('Balance por clase:', torch.bincount(train_pack['y']).tolist())

# Channel normalization only if tensors were NOT pre-normalized during export.
# With already_normalized=True the tensors already have per-sample z-score (notebook 3.2),
# therefore NO second dataset-level normalization is applied.
if not already_normalized:
    MIN_STD = 1e-6

    def compute_channel_stats(x: torch.Tensor):
        x = x.float()
        mean = x.mean(dim=(0, 2, 3))
        std = x.std(dim=(0, 2, 3)).clamp_min(MIN_STD)
        return mean, std

    CHANNEL_MEAN, CHANNEL_STD = compute_channel_stats(train_pack['x'])
    print(f'CHANNEL_MEAN={CHANNEL_MEAN}, CHANNEL_STD={CHANNEL_STD}')


In [ ]:
class TensorPackDataset(Dataset):
    """Dataset de tensores pre-exportados con z-score por muestra (notebook 3.2).

    Augmentation online: solo SpecAugment (FrequencyMasking + TimeMasking).
    Noise and shift were already applied during export for 'surprised',
    so they are NOT repeated here to avoid double augmentation.
    Augmentation is applied ONLY to the training split (augment=True).
    """
    def __init__(self, pack, augment=False):
        self.x = pack['x'].float()  # [N, 3, n_mels, n_frames]
        self.y = pack['y'].long()
        self.augment = augment

        # SpecAugment only for train (input size: 60 mel bins x 51 frames)
        if augment:
            self.spec_aug = nn.Sequential(
                T.FrequencyMasking(freq_mask_param=3),
                T.TimeMasking(time_mask_param=2)
            )

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        x = self.x[idx]
        y = self.y[idx]

        if self.augment:
            # SpecAugment expects [C, F, T] or [B, C, F, T]
            x = self.spec_aug(x)

        return x, y

In [ ]:
# We only augment the TRAIN set. Val and Test must remain clean.
#
# Note about WeightedRandomSampler:
# The train dataset already includes extra 'surprised' samples (export augmentation
# in notebook 3.2). Enabling WeightedRandomSampler would duplicate the oversampling of
# that class. That is why use_weighted_sampler=False is the safe default value.
# If tensors are regenerated WITHOUT offline augmentation, it can be enabled with True.

def build_dataloaders(batch_size=BATCH_SIZE, use_weighted_sampler=False):
    pin = torch.cuda.is_available()
    num_workers = 2 if torch.cuda.is_available() else 0

    train_ds = TensorPackDataset(train_pack, augment=True)
    val_ds = TensorPackDataset(val_pack, augment=False)
    test_ds = TensorPackDataset(test_pack, augment=False)

    sampler = None
    shuffle_train = True
    if use_weighted_sampler:
        sample_weights = CLASS_WEIGHTS[train_pack['y']]
        sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
        shuffle_train = False

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, shuffle=shuffle_train,
                              num_workers=num_workers, pin_memory=pin)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin)

    return train_loader, val_loader, test_loader, class_names, CLASS_WEIGHTS

train_loader, val_loader, test_loader, class_names, class_weights = build_dataloaders()


In [ ]:
# Weigths and biases | Configuration:

USE_WANDB = True
WANDB_PROJECT = "tcoe-transfer-no-aug"
WANDB_GROUP = "resnet18-and-effb0"

# Hiper parametros fijos


EPOCHS = 50



# Local directory for the best checkpoints:

CHECKPOINT_DIR = '/content/checkpoints_transfer_no_aug'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Matriz de experimentos:




### Model Factory

ResNet18 y Efficientnet-b0

In [ ]:
class TensorModelFactory:
  @staticmethod
  def get_model(model_name: str, num_classes: int, freeze_base: bool=True):
    if model_name == 'resnet18':
      model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
      # Adaptation for small input [3,60,51]
      model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
      model.maxpool = nn.Identity()
      if freeze_base:
        for param in model.parameters():
          param.requires_grad = False

      in_features = model.fc.in_features
      model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_features, num_classes))

    elif model_name == 'efficientnet_b0':
      model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)


      if freeze_base:
          for p in model.parameters():
              p.requires_grad = False

      in_f = model.classifier[1].in_features
      model.classifier = nn.Sequential(
          nn.Dropout(0.4),
          nn.Linear(in_f, num_classes)
      )

    else:
        raise ValueError(f"Model not supported for tensor pipeline: {model_name}")

    return model.to(device)

### Neural network training

Function to train with ResNet or EfficientNet-B0. The model training functions are very similar; in them we can configure network parameters, for example:
* Directory where we save the models `save_path`,
* Number of epochs `epochs`,
* Learning rate `lr`,
* Number of attempts before stopping training `patience`,
* Weight configuration `weight_decay`.

In [ ]:
def set_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def cleanup_state():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


HEAD_PREFIXES = ('fc', 'classifier', 'head')


def split_parameters(model, head_prefixes=HEAD_PREFIXES):
    head_params, backbone_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith(head_prefixes):
            head_params.append(param)
        else:
            backbone_params.append(param)
    return backbone_params, head_params


def build_optimizer(model, lr, backbone_lr=None, weight_decay=1e-2):
    backbone_params, head_params = split_parameters(model)
    if not backbone_params or backbone_lr is None:
        return optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                           lr=lr, weight_decay=weight_decay)

    return optim.AdamW(
        [
            {'params': backbone_params, 'lr': backbone_lr},
            {'params': head_params, 'lr': lr}
        ],
        weight_decay=weight_decay
    )


def train_model_tensor(
    model,
    train_loader,
    val_loader,
    save_path,
    epochs=50,
    lr=1e-4,
    backbone_lr=None,
    patience=5,
    weight_decay=1e-2,
    label_smoothing=0.05,
    max_grad_norm=1.0,
    class_weights=None
):
    if class_weights is not None:
        class_weights = class_weights.to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
    optimizer = build_optimizer(model, lr=lr, backbone_lr=backbone_lr, weight_decay=weight_decay)

    # scheduler maximiza F1 macro
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3
    )
    scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

    # best checkpoint por F1 macro
    best_val_f1 = -1.0
    trigger = 0

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0

        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                logits = model(x)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            if max_grad_norm is not None:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * x.size(0)
            preds = logits.argmax(dim=1)
            train_correct += (preds == y).sum().item()
            train_total += y.size(0)

        train_loss /= max(train_total, 1)
        train_acc = train_correct / max(train_total, 1)

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        all_val_y, all_val_p = [], []

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                    logits = model(x)
                    loss = criterion(logits, y)

                val_loss += loss.item() * x.size(0)
                preds = logits.argmax(dim=1)

                val_correct += (preds == y).sum().item()
                val_total += y.size(0)
                all_val_y.extend(y.cpu().numpy())
                all_val_p.extend(preds.cpu().numpy())

        val_loss /= max(val_total, 1)
        val_acc = val_correct / max(val_total, 1)
        val_f1_macro = f1_score(all_val_y, all_val_p, average='macro', zero_division=0)

        scheduler.step(val_f1_macro)
        lr_now = optimizer.param_groups[0]['lr']

        print(f"[{epoch+1:02d}/{epochs}] lr={lr_now:.1e} | "
              f"train_loss={train_loss:.4f} acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} acc={val_acc:.4f} f1m={val_f1_macro:.4f}")

        if val_f1_macro > best_val_f1:
            best_val_f1 = val_f1_macro
            torch.save(model.state_dict(), save_path)
            trigger = 0
        else:
            trigger += 1
            print(f"Early stop activado: {trigger}/{patience}")
            if trigger >= patience:
                print("Early stopping.")
                break

    return save_path


Training run

In [ ]:
# State cleanup to avoid memorized results
set_global_seed(42)
cleanup_state()

In [ ]:
MODELS_SAVE_DIR = "/content/saved_models"
os.makedirs(MODELS_SAVE_DIR, exist_ok=True)
MODEL_ARCH_1 = 'efficientnet_b0'
exp_name_b0 = f"tensor{MODEL_ARCH_1}"

train_loader, val_loader, test_loader, class_names, class_weights = build_dataloaders()

phase1_path = os.path.join(MODELS_SAVE_DIR, f"{exp_name_b0}efficient_fase1.pth")

model = TensorModelFactory.get_model(MODEL_ARCH_1, num_classes=len(class_names), freeze_base=True)

best_phase1_eff = train_model_tensor(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    save_path=phase1_path,
    epochs=30,
    lr=1e-3,
    patience=4,
    class_weights=class_weights
)

print("Phase 1 best model:", best_phase1_eff)

In [ ]:
# Descongelamiento de las capas de la CNN
phase2_path = os.path.join(MODELS_SAVE_DIR, f"{exp_name_b0}_fase2.pth")

model.load_state_dict(torch.load(best_phase1_eff, map_location=device, weights_only=True))
for p in model.parameters():
    p.requires_grad = True

best_phase2_eff = train_model_tensor(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    save_path=phase2_path,
    epochs=50,
    lr=1e-5,
    backbone_lr=1e-5,
    weight_decay=1e-3,
    patience=5,
    class_weights=class_weights
)

print("Phase 2 best efficient model:", best_phase2_eff)


Model evaluation

In [ ]:

def evaluate_tensor_model_eff(model_arch, model_path, test_loader, class_names):
    model = TensorModelFactory.get_model(model_arch, num_classes=len(class_names), freeze_base=False)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()

    all_y, all_p = [], []

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                logits = model(x)
            preds = logits.argmax(dim=1)

            all_y.extend(y.cpu().numpy())
            all_p.extend(preds.cpu().numpy())

    acc = accuracy_score(all_y, all_p)
    f1m = f1_score(all_y, all_p, average='macro', zero_division=0)
    print(f"Test Accuracy: {acc:.4f} | Test F1 macro: {f1m:.4f}\n")
    print(classification_report(all_y, all_p, target_names=class_names, zero_division=0))

    cm = confusion_matrix(all_y, all_p)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - {model_arch} (tensor)")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

    del model
    torch.cuda.empty_cache()
    gc.collect()

evaluate_tensor_model_eff(MODEL_ARCH_1, best_phase2_eff, test_loader, class_names)

### Fine-tuning with ResNet18

In [ ]:
set_global_seed(42)
cleanup_state()

In [ ]:
MODELS_SAVE_DIR = "/content/saved_models"
os.makedirs(MODELS_SAVE_DIR, exist_ok=True)
MODEL_ARCH_2 = 'resnet18'
exp_name_18 = f"tensor_{MODEL_ARCH_2}"

train_loader, val_loader, test_loader, class_names, class_weights = build_dataloaders()

phase1_path = os.path.join(MODELS_SAVE_DIR, f"{exp_name_18}resnet_fase1.pth")

model = TensorModelFactory.get_model(MODEL_ARCH_2, num_classes=len(class_names), freeze_base=True)

best_fase1_18 = train_model_tensor(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    save_path=phase1_path,
    epochs=20,
    lr=1e-3,
    patience=6,
    class_weights=class_weights
)

print("Phase 1 best model:", best_fase1_18)


In [ ]:
phase2_path = os.path.join(MODELS_SAVE_DIR, f"{exp_name_18}_phase2_unfrozen.pth")

model.load_state_dict(torch.load(best_fase1_18, map_location=device, weights_only=True))
for p in model.parameters():
    p.requires_grad = True

best_fase2_18 = train_model_tensor(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    save_path=phase2_path,
    epochs=50,
    lr=1e-5,
    backbone_lr=1e-5,
    weight_decay=1e-3,
    patience=5,
    class_weights=class_weights
)

print("Phase 2 best ResNet model:", best_fase2_18)

In [ ]:
def evaluate_tensor_model(model_arch, model_path, test_loader, class_names):
    model = TensorModelFactory.get_model(model_arch, num_classes=len(class_names), freeze_base=False)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()

    all_y, all_p = [], []

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                logits = model(x)
            preds = logits.argmax(dim=1)

            all_y.extend(y.cpu().numpy())
            all_p.extend(preds.cpu().numpy())

    acc = accuracy_score(all_y, all_p)
    f1m = f1_score(all_y, all_p, average='macro', zero_division=0)
    print(f"Test Accuracy: {acc:.4f} | Test F1 macro: {f1m:.4f}\n")
    print(classification_report(all_y, all_p, target_names=class_names, zero_division=0))

    cm = confusion_matrix(all_y, all_p)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - {model_arch} (tensor)")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

    del model
    torch.cuda.empty_cache()
    gc.collect()

evaluate_tensor_model(MODEL_ARCH_2, best_fase2_18, test_loader, class_names)

In [ ]:
set_global_seed(42)
cleanup_state()

In [ ]:
import shutil
# Save the models for the next stage (late fusion)
output_filename = '/content/saved_models'
shutil.make_archive(output_filename, 'zip', MODELS_SAVE_DIR)

# Rename the zip file so it has the .zip extension
# This is a workaround for make_archive, which does not add the extension by default if base_name already looks like one.
import os
os.rename('/content/saved_models.zip', '/content/saved_models_final.zip')

!cp /content/saved_models_final.zip /content/drive/MyDrive/saved_models_img_and_tensor.zip